# TP 9-10 Requêtes opendata et similarité

date de modification 09/04/2026


Objectif : Comprendre les notions de similarité et gestion de vecteurs



# Préparation

**Copier** ce notebook dans votre drive et l'exécuter dans l'environnement colab de google


In [2]:
!pip install -q SPARQLWrapper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 13.6 MB/s eta 0:00:00


lib pour les embeddings

In [3]:
!pip install -q hnswlib

# embeddings
# !pip install -Uq pymilvus
# !pip install -q pymilvus[model]

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


### Importer le pilote pour accéder aux sources

In [4]:
from SPARQLWrapper import SPARQLWrapper, JSON, CSV
print("pilote importé")

pilote importé


### Manipuler le résultat d'une requête
Fonctions pour afficher le résultat d'une requête

In [5]:
import pandas as pd

#pd.set_option('max_columns', None)
pd.set_option('max_colwidth', None)


# Conversion de JSON vers un dataframe
def get_result(results, limit=100):
  # Get the variable names from the results
  variable_names = results["head"]["vars"]
  # print("variables projetées:", variable_names)

  # Extract tuples using the variable names, taking into account optionnal bindings that may be NULL
  tableau = []
  for result in results["results"]["bindings"][:limit]:
      tableau.append([result.get(var, {}).get("value", "") for var in variable_names])
  return pd.DataFrame(tableau, columns = variable_names)


def query_source(source, requete,  limit=100):
  # invoque le service SPARQL distant
  source.setQuery(requete)

  # formate le résultat en json
  res = source.query().convert()

  # convertit le résultat en un dataframe
  return get_result(res, limit)

print ("fonctions définies")

fonctions définies


### Importer le pilote pour accéder aux sources

In [6]:
from SPARQLWrapper import SPARQLWrapper
print("pilote importé")

pilote importé


## Accès aux sources

Une source est caratérisée par une URL d'accès. Pour éviter une requête transférant trop de données, une source peut ne retourner que les N premiers tuples du résultat d'une requête (N dépend de la source).
Un ensemble de prefixe est défini pour simplifier l'expression des requêtes.

### Accès DBLP

In [7]:
# Set up the SPARQL endpoint
DBLP = SPARQLWrapper("https://sparql.dblp.org/sparql")
DBLP.setReturnFormat(JSON)

DBLP_PREFIX = """
PREFIX dblp: <https://dblp.org/rdf/schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
"""

### Accès à la BNF

In [8]:
# Connexion à la BNF
BNF = SPARQLWrapper("https://data.bnf.fr/sparql")
BNF.setReturnFormat(JSON)

BNF_prefix = """
PREFIX dc: <http://purl.org/dc/terms/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX rda: <http://rdaregistry.info/Elements/w/#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

PREFIX bnf: <http://data.bnf.fr/ontology/bnf-onto/>

PREFIX bio: <http://vocab.org/bio/0.1/>
PREFIX voc2: <http://rdvocab.info/ElementsGr2/>

PREFIX bnfwf: <http://data.bnf.fr/vocabulary/work-form/>

"""
# print(BNF_prefix)

### Accès à DBPEDIA France

In [9]:
# Connexion à DBPEDIA France.
DBPEDIA = SPARQLWrapper("https://fr.dbpedia.org/sparql")
DBPEDIA.setReturnFormat(JSON)

DBPEDIA_prefix =  """
PREFIX dc:   <http://purl.org/dc/terms/>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>

PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

PREFIX dbo:  <http://dbpedia.org/ontology/>

# Dans cette source, les noms de propriétés sont en français
PREFIX dbp:  <http://fr.dbpedia.org/property/>
"""
# print(DBPEDIA_prefix)

# Requêtes DBLP

Schéma global

**Author** (authorID, name)

**Publication** (pubID, title, year)

**Publish** (authorID, pubID, venue)


## DBLP1 : liste des auteurs ayant publié à partir de 2024 dans la conférence SIGMOD

In [11]:
requete = DBLP_PREFIX + """
SELECT distinct ?author ?name
WHERE {
  ?publication dblp:authoredBy ?author .
  ?author dblp:creatorName ?name .
  ?publication dblp:title ?title .
  ?publication dblp:publishedIn ?venue .
  ?publication dblp:yearOfPublication ?year
  FILTER (?year >= "2024"^^xsd:gYear && CONTAINS(?venue,"SIGMOD Conference") )
}
# LIMIT 20
"""

# Execute the query and get results
df = query_source(DBLP, requete, limit=1000)
df

,author,name
0,https://dblp.org/pid/00/232,Alan Halverson
1,https://dblp.org/pid/00/3717,Stefan C. Müller
2,https://dblp.org/pid/00/8454-4,Yujie Wang
3,https://dblp.org/pid/01/24-6,Ling Yang
4,https://dblp.org/pid/01/7213,Mingjie Tang
...,...,...
995,https://dblp.org/pid/67/5612,Mikael Ronström
996,https://dblp.org/pid/67/6944,Tobias Schmidt
997,https://dblp.org/pid/67/9320,Sebastian Hillig
998,https://dblp.org/pid/68/10080,Wenbo Sun


## DBLP2: les publications d'un auteur donné, postérieures à une date donnée

C'est un requête paramétrée sur l'id d'un auteur

In [ ]:
# J. Leskovec
# AUTHOR = "<https://dblp.org/pid/l/JureLeskovec>"

# I. Trummer
AUTHOR = "<https://dblp.org/pid/85/9079>"

MIN_YEAR = "2000"

requete = DBLP_PREFIX + """
SELECT ?publication ?title ?year ?venue
WHERE {
  BIND( {AUTHOR} as ?author).
  BIND( "{MIN_YEAR}"^^xsd:gYear as ?min_year).

  ?publication dblp:authoredBy ?author .
  ?publication dblp:title ?title .
  ?publication dblp:yearOfPublication ?year .
  ?publication dblp:publishedIn ?venue .
  FILTER (?year >= ?min_year)
}
ORDER BY desc(?year)
LIMIT 100
"""

requete = DBLP_PREFIX + """
SELECT distinct ?venue
WHERE {
  BIND( {AUTHOR} as ?author).
  BIND( "{MIN_YEAR}"^^xsd:gYear as ?min_year).

  ?publication dblp:authoredBy ?author .
  ?publication dblp:title ?title .
  ?publication dblp:yearOfPublication ?year .
  ?publication dblp:publishedIn ?venue .
  FILTER (?year >= ?min_year)
}
ORDER BY desc(?year)
LIMIT 100
"""
requete = requete.replace("{AUTHOR}", AUTHOR)
requete = requete.replace("{MIN_YEAR}", MIN_YEAR)

# Execute the query and get results
query_source(DBLP, requete)

,venue
0,SIGMOD Conference Companion
1,Proc. ACM Manag. Data
2,RLJ
3,Q-DATA@SIGMOD
4,VLDB Workshops
5,CoRR
6,AAAI
7,ACM Trans. Database Syst.
8,IEEE Data Eng. Bull.
9,ICDE


## DBLP3: les coauteurs d'un auteur donné, triés par nombre de publications communes

In [ ]:
AUTHOR = "<https://dblp.org/pid/00/232>"

MIN_YEAR = "2000"

requete = DBLP_PREFIX + """
SELECT ?coauthor (COUNT(?publication) AS ?numPublications)
WHERE {
  BIND( {AUTHOR} as ?author).
  BIND( "{MIN_YEAR}"^^xsd:gYear as ?min_year).

  ?publication dblp:authoredBy  ?author.
  ?publication dblp:authoredBy ?coauthor .
  ?publication dblp:yearOfPublication ?year .
  FILTER (?author != ?coauthor && ?year >= ?min_year)
}
GROUP BY  ?coauthor
ORDER BY desc(?numPublications)
LIMIT 10
"""
requete = requete.replace("{AUTHOR}", AUTHOR)
requete = requete.replace("{MIN_YEAR}", MIN_YEAR)

# Execute the query and get results
query_source(DBLP, requete)

,coauthor,numPublications
0,https://dblp.org/pid/d/DJDeWitt,6
1,https://dblp.org/pid/95/4983,4
2,https://dblp.org/pid/p/JMPatel,4
3,https://dblp.org/pid/n/JeffreyFNaughton,4
4,https://dblp.org/pid/51/792-2,3
5,https://dblp.org/pid/33/5504,3
6,https://dblp.org/pid/84/3247,3
7,https://dblp.org/pid/367/4168,2
8,https://dblp.org/pid/367/4148,2
9,https://dblp.org/pid/47/3052,2


## DBLP4: les publications récentes dont le titre contient un certain terme

In [ ]:
TERM = "big data"

requete = DBLP_PREFIX + """
SELECT ?publication ?title ?year ?venue
WHERE {
  ?publication dblp:publishedIn ?venue .
  ?publication dblp:yearOfPublication ?year .
  ?publication dblp:title ?title .
  FILTER (?year >= "2024"^^xsd:gYear && CONTAINS(?title," {TERM} ")  )
}
LIMIT 10
"""

requete = requete.replace("{TERM}", TERM)

# Execute the query and get results
df = query_source(DBLP, requete)
df

,publication,title,year,venue
0,https://dblp.org/rec/conf/acis/ShahNF24,"Explaining the possible negative effects of big data through the interplay between big data analytics, governance, and privacy: A Socio-technical analysis.",2024,ACIS
1,https://dblp.org/rec/conf/ant/RuizSFLMC25,Customer segmentation analysis with big data in health services companies in Colombia: case study.,2025,ANT/EDI40
2,https://dblp.org/rec/conf/bde/LiuHL24,The application of big data to the conservation of wildlife in highland areas.,2024,BDE
3,https://dblp.org/rec/conf/bde/ZhaoZZ24,Behavior detection of mobile phone use at work based on big data and optimized YOLOv6.,2024,BDE
4,https://dblp.org/rec/conf/bdsic/JiangWLML24,Research on fish disease detection based on big data and deep learning.,2024,BDSIC
5,https://dblp.org/rec/conf/bdsic/LiaoY24,DvaDRP: a dual variational autoencoder based on big data for drug response prediction.,2024,BDSIC
6,https://dblp.org/rec/conf/bdsic/ZhangSZHYLCZ25,A fast and parallel framework for massive sensitive data detection based on big data technologies and large language models (LLMs).,2025,BDSIC
7,https://dblp.org/rec/conf/bic/ChenS24,Design of artificial intelligence auxiliary system for elderly care service based on big data analysis.,2024,BIC
8,https://dblp.org/rec/conf/caibda/ZhangC24,"Artificial intelligence, big data and algorithms make it possible for stakeholders to build smart tourism destinations: take Tianzhu Mountain Scenic Area as an example.",2024,CAIBDA
9,https://dblp.org/rec/conf/ciaw/Vysotska24,"Computational intelligence technology for Ukrainian language textual content processing based on big data analysis, NLP and machine learning.",2024,CIAW


# Exercice 1 : requêtes sur DBLP

### a) Associer auteur et publication

A l'aide des requêtes précédentes DBLP1 et DBLP2. Créer une table Publie(authorID, name, title) associant le titre d'une publication avec un auteur, tel que
*   authorID existe dans DBLP1
*   title  existe dans DBLP2



In [1]:
requete2 = DBLP_PREFIX + """
SELECT distinct ?venue
WHERE {
  BIND( {AUTHOR} as ?author).
  BIND( "{MIN_YEAR}"^^xsd:gYear as ?min_year).

  ?publication dblp:authoredBy ?author .
  ?publication dblp:title ?title .
  ?publication dblp:yearOfPublication ?year .
  ?publication dblp:publishedIn ?venue .
  FILTER (?year >= ?min_year)
}
ORDER BY desc(?year)
LIMIT 100
"""

AUTHOR = "<https://dblp.org/pid/85/9079>"
MIN_YEAR = "2000"

requete2 = requete2.replace("{AUTHOR}", AUTHOR)
requete2 = requete2.replace("{MIN_YEAR}", MIN_YEAR)

requete = """
SELECT DISTINCT ?author ?name ?title
WHERE {
  ?publication dblp:authoredBy ?author .
  ?author dblp:creatorName ?name .
  ?publication dblp:title ?title .
  ?publication dblp:publishedIn ?venue .
  ?publication dblp:yearOfPublication ?year .

  FILTER (?year >= "2024"^^xsd:gYear)
  FILTER CONTAINS(?venue, "SIGMOD Conference")
}
ORDER BY ?name
"""



query_source(DBLP, requete)
# Execute the query and get results

resultats = []

requete1 = DBLP_PREFIX + """
SELECT distinct ?author ?name
WHERE {
  ?publication dblp:authoredBy ?author .
  ?author dblp:creatorName ?name .
  ?publication dblp:title ?title .
  ?publication dblp:publishedIn ?venue .
  ?publication dblp:yearOfPublication ?year
  FILTER (?year >= "2024"^^xsd:gYear && CONTAINS(?venue,"SIGMOD Conference") )
}
# LIMIT 20
"""

df1 = query_source(DBLP, requete1)

for _, row in df1.iterrows():
    author = f"<{row['author']}>"
    name = row['name']

    requete2 = DBLP_PREFIX + """
    SELECT ?title
    WHERE {
      BIND( {AUTHOR} as ?author).
      ?publication dblp:authoredBy ?author .
      ?publication dblp:title ?title .
    }
    """

    requete2 = requete2.replace("{AUTHOR}", author)

    df_pub = query_source(DBLP, requete2)

    for _, pub in df_pub.iterrows():
        resultats.append({
            "authorID": author,
            "name": name,
            "title": pub["title"]
        })

NameError: name 'DBLP_PREFIX' is not defined

### b) Co auteurs

En utilisant des requêtes parmi DBLP1 à DBLP4, créer un table **Coauthor**(authorID1, name1, autorID2, name2, nbPubli) telle que les deux auteurs ont au moins une publication commune depuis l'année 2010,  et authorID1 est dans DBLP1.

Indication: proposer 2 solutions:
*   avec DBLP1 et DBLP2
*   avec DBLP1 et DBLP3


# Exercice 2 : Accès par morceau au résultat d'une requête





Un utilisateur veut accéder aux données de manière progressive.

On considère  la requête qui affiche le nom des personnes existant dans dbpedia.

On peut obtenir un certain morceau du résultat d'une requete. Le morceau commençant au tuple i et contenant N tuples est préciser avec les mots-clés `OFFSET i LIMIT N`.




In [ ]:
# import time

# requete = DBPEDIA_prefix + """

# SELECT distinct ?name ?p
# WHERE {
#   ?p rdf:type dbo:Person .
#   ?p foaf:name ?name .
#    FILTER(str(?name) != "")
# }
# offset .... limit .......
# """

# start_time = time.time()
# # exécution de la requête
# df = query_source(DBPEDIA, requete)

# duree = int ( (time.time() - start_time) * 1000)
# print(f"la requête dure {duree} ms")

# df

## Questions
1) Définir une fonction traite_requete_par_bloc(requete) qui retourne un generateur sur une requete de telle sorte que la requete soit lue par morceaux de N tuples et que seuls les morceaux permettant de produire les M lignes demandées par l'utlisateur soit lus. On ne connait pas à l'avance le nombre M de lignes que va lire l'utilisateur. On a M > N.
Indication utiliser le mot clé python `yield` pour retourner un générateur.

In [10]:
requete = DBPEDIA_prefix + """
SELECT distinct ?name ?p
WHERE {
  ?p rdf:type dbo:Person .
  ?p foaf:name ?name
  FILTER(str(?name) != "")
}
"""

# définition d'un générateur
def traite_requete_par_bloc(requete):
  offset = 0
  limit = 5
  while(True):
    custom_requete = requete + f"\nOFFSET {offset} \nLIMIT {limit}"
    df = query_source(DBPEDIA, custom_requete)
    offset += limit

    if df.empty: break

    yield df

# lecture par l'utilisateur
a_lire = 22
lu = 0
for a in traite_requete_par_bloc(requete):
  print(a)
  lu+=1
  if(lu >= a_lire):
    break

print()
if(lu < a_lire):
  print(f"Attention !!!! l'utilisateur n'a obtenu que {lu} lignes sur les {a_lire} à lire")
else:
  print(f"OK, l'utilisateur a obtenu {lu} lignes sur les {a_lire} à lire")


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1010)>

2) Proposer une solution pour accéder de manière progressive au résultat d'une requête répartie entre 2 sites.

In [ ]:
# à compléter

# Exerice 3 : Cache de requêtes
1) Proposer des améliorations des requêtes précédentes afin d'éviter de poser plusieurs fois une même requête. On suppose que le résultat de la requête ne change pas entre deux invocations de la même requête.

2) Cache avec une capacité limitée : considérer qu'on peut conserver en cache le résultat de C requêtes (C=10) au maximum

In [ ]:
# A compléter

# Exercice 4 : Recherche de publications similaires avec embeddings

On veut exprimer la requête : étant donné une phrase *P*, quelles sont les *k* publications de la source DBLP ayant un titre le plus similaire à *P* et dont l'année > *YEAR_MIN* et publiées dans la conférence *CONFERENCE* ?

Afficher une table (publication, titre, année, distance) avec distance étant la distance cosinus entre une publication et la phrase requête.


Indications:
*    **requêter** la source pour obtenir les publications qui satisfont le critère sur l'année et la conférence.

*   pour chaque publication, inférer un **embedding** de son titre.

*   **indexer** les publications selon leur embedding en utilisant l'index  HNSW : https://zilliz.com/learn/learn-hnswlib-graph-based-library-for-fast-anns

*   Inférer l'embedding de la phrase P.

*   Interroger l'index pour connaître les **k plus proches** embeddings  de l'embedding de la phrase P.

Installer un encodeur pour inférer des embeddings

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Charger un modèle pré-entraîné (ex: 'all-MiniLM-L6-v2' est rapide et efficace)
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Obtenir les publications tq année >YEAR_MIN et conférence = CONFERENCE

In [23]:
YEAR_MIN = 2023

CONFERENCE="SIGMOD Conference"

requete = DBLP_PREFIX + """
# SELECT  (count(?publication) as ?nb)
# WHERE {

SELECT ?publication ?title ?year
WHERE {
  ?publication dblp:publishedIn ?venue .
  ?publication dblp:yearOfPublication ?year .
  ?publication dblp:title ?title .
  FILTER (?year >= "{YEAR_MIN}"^^xsd:gYear && CONTAINS(?venue,"{CONFERENCE}")  )
}

# }
"""

requete = requete.replace("{YEAR_MIN}", str(YEAR_MIN))
requete = requete.replace("{CONFERENCE}", CONFERENCE)

# Execute the query and get results
publications = query_source(DBLP, requete, limit=1000)
publications

,publication,title,year
0,https://dblp.org/rec/conf/sigmod/0001000TC0X25,Authenticating Multi-Chain Queries: Verifiable Virtual Filesystem Is All You Need.,2025
1,https://dblp.org/rec/conf/sigmod/000123,Mixed Methods Machine Learning.,2023
2,https://dblp.org/rec/conf/sigmod/000123a,DEI Perspectives in Information Technology Education.,2023
3,https://dblp.org/rec/conf/sigmod/0001FDZ24,Beyond Bloom: A Tutorial on Future Feature-Rich Filters.,2024
4,https://dblp.org/rec/conf/sigmod/0001HSV23,Seventh Workshop on Data Management for End-to-End Machine Learning (DEEM).,2023
...,...,...,...
286,https://dblp.org/rec/conf/sigmod/ZhuSA23,Acheron: Persisting Tombstones in LSM Engines.,2023
287,https://dblp.org/rec/conf/sigmod/ZhuSKMMHCTH0KKS25,Rockhopper: A Robust Optimizer for Spark Configuration Tuning in Production Environment.,2025
288,https://dblp.org/rec/conf/sigmod/ZhuTCKAACCCDFGH23,Towards Building Autonomous Data Services on Azure.,2023
289,https://dblp.org/rec/conf/sigmod/ZhuWDZ24,Learned Query Optimizer: What is New and What is Next.,2024


pour chaque publication, inférer un **embedding** de son titre.

*   List item
*   List item




In [17]:
# les titres à encoder
titres = np.array(publications['title']).astype(str).tolist()
num_elements = len(titres)
print('nb de titres de publication :', num_elements)

# Inférer les embeddings
embeddings = model.encode(titres, convert_to_numpy=True)
# print('embeddings shape', embeddings.shape)

dim = embeddings.shape[1]
print("dimension d'un embedding:", dim) #384 dimensions par vecteur

# les id des titres dans [0, num_elements[
ids = np.arange(num_elements)

nb de titres de publication : 291
dimension d'un embedding: 384


**indexer** les publication selon leur embedding en utilisant l'index  HNSW

On commencer par créer l'index

In [18]:
import hnswlib

# Declaring index
index = hnswlib.Index(space = 'cosine', dim = dim) # possible options are l2, cosine or ip

# Initializing index - the maximum number of elements should be known beforehand
index.init_index(max_elements = 10000, ef_construction = 25, M = 8)

# Controlling the recall by setting ef:
index.set_ef(10) # ef should always be > k

print("index créé")

index créé


On insère les embeddings dans l'index

In [19]:
index.add_items(embeddings, ids)
print(f'added {len(ids)} elts in the index')

added 291 elts in the index


inférer l'embedding de la phrase requête P


In [45]:
# les requêtes
phrase_requetes= titres[:2]
# phrase_requetes= ["distributed query processing"]

query_vectors = model.encode(phrase_requetes, convert_to_numpy=True)


retrouver dans l'index les *k* titres similaires à P, ainsi que les publications associées

In [46]:
# Query dataset, k - number of the closest elements (returns 2 numpy arrays)
labels, distances = index.knn_query(query_vectors, k = 3)

# print(labels)
for i, (L, D) in enumerate(zip(labels, distances)):
  print("requête:", phrase_requetes[i])
  print("titres similaires:")
  for l, d in zip(L, D):
    # print(titres[l], "distance:", abs(round(d,3)))
    print(tuple(publications.iloc[l]), "distance:", abs(round(d,3)))
  print()

requête: Authenticating Multi-Chain Queries: Verifiable Virtual Filesystem Is All You Need.
titres similaires:
('https://dblp.org/rec/conf/sigmod/0001000TC0X25', 'Authenticating Multi-Chain Queries: Verifiable Virtual Filesystem Is All You Need.', '2025') distance: 0.0
('https://dblp.org/rec/conf/sigmod/DinhOY23', 'First Workshop on Verifiable Database Systems.', '2023') distance: 0.434
('https://dblp.org/rec/conf/sigmod/TianLZTHCY0JZJ24', 'LETUS: A Log-Structured Efficient Trusted Universal BlockChain Storage.', '2024') distance: 0.499

requête: Mixed Methods Machine Learning.
titres similaires:
('https://dblp.org/rec/conf/sigmod/000123', 'Mixed Methods Machine Learning.', '2023') distance: 0.0
('https://dblp.org/rec/conf/sigmod/CongYZ24', 'Machine Learning for Databases: Foundations, Paradigms, and Open problems.', '2024') distance: 0.507
('https://dblp.org/rec/conf/sigmod/Baeza-Yates24', 'The Limitations of Data, Machine Learning and Us.', '2024') distance: 0.591



## Question 1
Amliorer la solution afin de créer un dataframe pour le résultat qui doit être une table (publicationID, titre, année, distance).

In [47]:
rows = []

for i, (L, D) in enumerate(zip(labels, distances)):
    for l, d in zip(L, D):
        rows.append({
            "publicationID": publications.iloc[l]["publication"],
            "titre": publications.iloc[l]["title"],
            "annee": publications.iloc[l]["year"],
            "distance": abs(round(d,3)),
            "query": phrase_requetes[i]
        })

df_result = pd.DataFrame(rows)
display(df_result)

,publicationID,titre,annee,distance,query
0,https://dblp.org/rec/conf/sigmod/0001000TC0X25,Authenticating Multi-Chain Queries: Verifiable Virtual Filesystem Is All You Need.,2025,0.000,Authenticating Multi-Chain Queries: Verifiable Virtual Filesystem Is All You Need.
1,https://dblp.org/rec/conf/sigmod/DinhOY23,First Workshop on Verifiable Database Systems.,2023,0.434,Authenticating Multi-Chain Queries: Verifiable Virtual Filesystem Is All You Need.
2,https://dblp.org/rec/conf/sigmod/TianLZTHCY0JZJ24,LETUS: A Log-Structured Efficient Trusted Universal BlockChain Storage.,2024,0.499,Authenticating Multi-Chain Queries: Verifiable Virtual Filesystem Is All You Need.
3,https://dblp.org/rec/conf/sigmod/000123,Mixed Methods Machine Learning.,2023,0.000,Mixed Methods Machine Learning.
4,https://dblp.org/rec/conf/sigmod/CongYZ24,"Machine Learning for Databases: Foundations, Paradigms, and Open problems.",2024,0.507,Mixed Methods Machine Learning.
5,https://dblp.org/rec/conf/sigmod/Baeza-Yates24,"The Limitations of Data, Machine Learning and Us.",2024,0.591,Mixed Methods Machine Learning.


## Question 2 : persistence des embeddings
On veut conserver les embeddings calculés pour une série de requêtes.

On demande de ne jamais recalculer l'embedding d'un titre pour lequel on a déjà son embedding.

Proposer une solution pour tenir compte de cela. Montrer que votre solution fonctionne pour 2 requêtes consécutives. La 2eme requête contenant des tuples inclus dans le résultat de la 1ere requete.

In [55]:
phrase_requetes= titres[:2]
saved_phrases = {}

def getQueryVector(phrase_requetes):
  toQuery = []

  for phrase in phrase_requetes:
    if phrase not in saved_phrases:
      saved_phrases[phrase] = model.encode([phrase], convert_to_numpy=True)[0]
    else:
      print(f"Cache hit: {phrase}")

    toQuery.append(saved_phrases[phrase])

  return np.vstack(toQuery)

getQueryVector(phrase_requetes)
getQueryVector(phrase_requetes)

Cache hit: Authenticating Multi-Chain Queries: Verifiable Virtual Filesystem Is All You Need.
Cache hit: Mixed Methods Machine Learning.


array([[-6.75615743e-02,  2.64440626e-02, -5.59558384e-02,
        -3.40676717e-02, -6.06937148e-03, -5.27618937e-02,
         1.28388684e-02,  2.79377326e-02,  1.21353054e-02,
         4.10967245e-04,  1.41122350e-02,  2.64657754e-03,
         9.81585607e-02,  7.38769118e-03, -2.00218931e-02,
         1.32125488e-03, -8.81127734e-03,  8.24752301e-02,
         5.85627072e-02,  1.39832748e-02, -7.37126991e-02,
        -2.52825227e-02, -2.88805757e-02,  3.37678343e-02,
        -3.36080790e-02, -4.69700545e-02,  1.72078107e-02,
        -6.71722442e-02, -7.60860071e-02, -5.47771528e-02,
         1.41741401e-02, -2.78295558e-02, -1.11189507e-01,
         5.53778149e-02,  3.69436503e-03,  4.14228905e-03,
         8.13759714e-02, -6.54200315e-02, -1.16100088e-01,
        -9.54620987e-02, -1.46434214e-02,  1.26056047e-02,
        -8.66513625e-02,  2.36185221e-03, -9.69624892e-02,
        -4.81880940e-02,  4.05353270e-02,  3.49942856e-02,
        -1.56201487e-02,  2.24340595e-02, -1.16144400e-0

## Question 3 : recherche k-NN exhaustive

Soit V un tableau de vecteurs, p un vecteur requête et k un entier.

a) Ecrire une fonction NN(V, p, k) qui effectue la recherhe exhaustive des k NN de p parmi les V vecteurs.

Indication : comparer p avec **tous** les vecteurs de V.

b) Montrer que k-NN est plus lent que l'accès par index pour un ensemble V suffisamment grand. Vous pouvez générer V aléatoirement dans cette question.

c) Mettre en évidence l'aspect approximatif de la recherche ANN : montrer un cas où le résultat k-ANN ne contient pas tous les voisins k-NN.



In [ ]:
import sklearn
from sklearn.metrics.pairwise import cosine_distances


def cosine_distance(p, V):
  p_mat = p.reshape(1, -1)
  return cosine_distances(p_mat, V)[0]

# Exemple k-NN (exhaustif)

d = cosine_distance(..................   ..............)
indices_tries = ............

# Exercice 5 : Requête hybride avec embeddings  et SQL

DuckDB est tres efficace pour gérer des relations
mais ne manipule pas de données vectorielles et ne propose pas de recherche ANN (cf. cours 10).

On considère la table :

**Publication** (publicationID, year, venue, title)


On veut combiner une recherche ANN avec des requêtes SQL de sélection sur des données gérées dans DuckDB.

Soit la requête **R1** :

SELECT *, distance(p.title, "phrase recherchee") as distance

FROM Publications p

WHERE (year between 2022 and 2024) and venue like '%VLDB'

ORDER BY distance

LIMIT 10

La fonction distance(phrase1, phrase2) renvoie la distance cosinus entre les embeddings (vecteurs) de deux phrases.


## Preparation

In [ ]:
!pip install -q duckdb

In [ ]:
!pip install -q itables

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 18.7 MB/s eta 0:00:00


In [ ]:
# pour améliorer l'affichage des tableaux contenant le resultat d'une requete
from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)

La fonction **display** affiche les n premières lignes du résultat d'une requête

In [ ]:
def display(query, n=30):
    return query.limit(n).df()

In [ ]:
import duckdb

# option 1: la base créée sera persistente dans un fichier
# db = duckdb.connect('mabase')

# option 2: la base sera uniquement en mémoire. Elle sera supprimée en fin de session.
db = duckdb.connect(':memory:')

# vérifier que le service fonctionne en posant une première requête.
r = db.sql("SELECT 'hello' as col ")
display(r)

Loading ITables v2.7.3 from the internet... (need help?)


In [ ]:
def display_schema(table):
  return display(db.sql(f"""
    SELECT column_name, data_type
    FROM duckdb_columns
    WHERE table_name = '{table}'
  """))

# def display_all_views():
#   return display(db.sql("select view_name, sql, temporary from duckdb_views"))


def function_exists(name):
  existing_functions = db.sql("select distinct function_name, from duckdb_functions() WHERE function_type = 'scalar' ").df()
  return name in existing_functions.values

## Créer la table des Publications

Obtenir des données (exple des publication DBLP)

In [ ]:
grande_requete = DBLP_PREFIX + """
SELECT ?publication ?year ?venue ?title
WHERE {
  ?publication dblp:publishedIn ?venue .
  ?publication dblp:yearOfPublication ?year .
  ?publication dblp:title ?title .
  FILTER (    ?year >= "2010"^^xsd:gYear
           && ( CONTAINS(?venue,"SIGMOD")
                || CONTAINS(?venue,"VLDB")
                || CONTAINS(?venue,"EDBT")
                || CONTAINS(?venue,"ICDE")
              )
          )
}
ORDER BY desc(?year) ?title
LIMIT 1000
"""

# Executer la requete en plusieurs appels

# lire 1000 pulications par appel
MAX_RESULT = 1000

publications = query_source(DBLP, grande_requete, limit=MAX_RESULT)

# lire 3000 de publications au total
for i in range(1, 3):
  suite = query_source(DBLP, grande_requete + f" OFFSET {MAX_RESULT*i}", limit=MAX_RESULT)
  publications = pd.concat([publications, suite])

print("nombre de publications", len(publications))
publications

nombre de publications 3000


Loading ITables v2.7.3 from the internet... (need help?)


Insérer les publications dans DuckDB

In [ ]:

# # Ajouter un numéro de publication pubID allant de 0 à n
# if 'pubID' not in publications.columns:
#   ...............

# # type int pour l'attribut year
# publications['year'] = publications['year'].astype(int)


# # importer les données dans une table DuckDB
# db.sql("drop table if exists publications")
# db.from_df(................).create(................)


# # on supopse que les publications ne sont plus dans l'application mais seulement dans DuckDB
# # del publications

# res = db.sql("SELECT * FROM publications ORDER BY pubID")
# display(res,5)

In [ ]:
display_schema('publications')

Loading ITables v2.7.3 from the internet... (need help?)


## Question 1: Indexation

Exporter les titres de la table publication, triés par pubID

In [ ]:
# titres_df = db.sql("SELECT ................").df()
# titres_publi = np.array(.................))

obtenir les embeddings de ces titres

In [ ]:
# Inférer les embeddings
import time
debut = time.time()
all_embeddings =.................
duree = time.time() - debut
print(f"Durée d'exécution : {duree:.3f} secondes, pour {len(titres_publi)} titres de publication")
# print('embeddings shape', embeddings.shape)

SyntaxError: invalid syntax (1819695403.py, line 4)

Indexer les embeddings

## Question 2 : Traiter la requête R1

Définir la fonction `requête_hybride1` qui traiter R1 en effectuant le maximum de traitements dans la base SQL.

Indication :  commencer par rechercher les K-ANN puis appliquer la condition de sélection


In [ ]:
def requête_hybride1(condition_selection, phrase):



  return res.df()

# Exemple
requête_hybride1("(year between 2020 and 2024) and venue like '%VLDB%'", "distributed query processing")

## Question 3: Fonction UDF

Définir une fonction UDF distance(p1,p2) retournant la distance cosinus entre les 2 vecteurs associées aux phrases p1 et p2

Proposer une solution pour traiter R1 en invoquant la fonction distance.

In [ ]:
# Exemple d'UDF

from duckdb.typing import FLOAT, VARCHAR

# Définition d'une UDF en Python
def carre(x):
    return x * x

# Enregistrement de l'UDF dans DuckDB
function_name = 'carre_UDF'
if function_exists(function_name):
  db.remove_function(function_name)
db.create_function(function_name, carre, [FLOAT], FLOAT)

print('UDF définie: ', function_name)

db.sql(f"SELECT carre_UDF(3)").df()

# FAQ: Questions diverses


## Mise au point des requêtes
Pour débugguer et **mettre au point vos requêtes** SPARQL, il est utile de les tester directement sur l'interface du point d'accès dédié de la source de données. L'avantage est que cela vous retourne un message indiquant plus précisément la nature et l'endroit où se trouve l'erreur dans la requête.

*   [DBLP](https://sparql.dblp.org/) et un [tuto](https://github.com/dblp/kg/wiki/dblp-KG-Tutorial)

*   [BNF](http://data.bnf.fr/sparql) voir des [exemples](https://api.bnf.fr/sparql-endpoint-de-databnffr)

*   [DBPedia](http://fr.dbpedia.org/sparql) et la liste de tous les [prefixes](http://fr.dbpedia.org/sparql?nsdecl)


## Autres sources de données
Autres sources à votre disposition pour proposer des requêtes multi-sources:

1.   [Mondial](http://www.semwebtech.org/mondial/10/sparql) : données géopolitiques. Attention le résulat des requêtes est en XML pas en JSON.
2.   [Persée](data.persee.fr) : données bibliographiques alignées avec plusieurs autres sources dont dbpedia. Voir des [exemples](http://data.persee.fr/ressources/requetes-exemples/)
3.   [BioPortal](http://sparql.bioontology.org)
4.    [IdRef](https://data.idref.fr/sparql) serveur d’autorité IdRef et les références bibliographiques en provenance du Sudoc.
5.   [Insee](https://rdf.insee.fr/sparql)
6.  [Geonames web services](https://www.geonames.org/export/ws-overview.html)


## SPARKLIS : aide à l'écriture de requêtes Sparql
Pour exprimer vos requêtes en *langage naturel* et les traduire automatiquement en SPARQL, voir l'appli [Sparklis](http://www.irisa.fr/LIS/ferre/sparklis/)


*   onglet **Sparklis** pour exprimer vos requêtes en langage naturel
*   onglet **YASGUI** pour voir la traduction en Sparql



## Nombre d'éléments dans le resultat d'une requête
Pour compter le nombre d'éléments d'une requête, utiliser la notation avec requête imbriquée dans le where :

SELECT (count(1) as ?nb)

WHERE { *sous-requete* }
